In [ ]:
import numpy as np
import networkx as nx
import random
import pandas as pd

# 设置随机种子以确保结果可复现
np.random.seed(42)
random.seed(42)

# 生成BA无标度网络并为每条边分配1到10之间的随机权重，返回邻接矩阵W
def generate_ba_scale_free_network(N, m):
    G = nx.barabasi_albert_graph(N, m)
    for u, v in G.edges():
        G[u][v]['weight'] = random.randint(1, 10)
    W = nx.to_numpy_array(G, weight='weight')
    return W

# 计算C矩阵
def calculate_C_matrix(W):
    N = W.shape[0]
    C = np.zeros_like(W, dtype=float)
    for i in range(N):
        row_sum = np.sum(W[i, :])
        if row_sum > 0:
            C[i, :] = W[i, :] / row_sum
    return C

# 计算M^1矩阵
def calculate_M1_matrix(C, k, p):
    N = C.shape[0]
    M1 = np.zeros_like(C, dtype=float)
    for i in range(N):
        for j in range(N):
            delta_ij = 1 if i == j else 0
            term1 = (1 - p)**2 * delta_ij * k[j]
            term2 = p * (1 - p) * k[j] * (C[i, j] + C[j, i])
            term3 = p**2 * k[j] * np.dot(C[i, :], C[:, j])
            M1[i, j] = term1 + term2 + term3
    return M1

# 计算M^2矩阵
def calculate_M2_matrix(C, n, p):
    N = C.shape[0]
    M2 = np.zeros_like(C, dtype=float)
    for i in range(N):
        for j in range(N):
            delta_ij = 1 if i == j else 0
            term1 = (1 - p)**2 * delta_ij * n[j] 
            term2 = p * (1 - p) * n[j] * (C[i, j] + C[j, i]) 
            term3 = p**2 * n[j] * np.dot(C[i, :], C[:, j])
            M2[i, j] = term1 + term2 + term3
    return M2

# 计算 \beta_c 并保存中间变量的计算步骤
def calculate_beta_c(M, mu, p, alpha, record):
    eigenvalues = np.linalg.eigvals(M)  # 计算M矩阵的特征值
    eigenvalues_real = np.real(eigenvalues)  # 取特征值的实部，避免复数值
    lambda_max = np.max(eigenvalues_real)    # 找到最大特征值
    beta_c = mu / lambda_max  # 根据公式计算beta_c
    
    # 将特征值、λmax、beta_c保存到记录中
    record.append({
        'p_move': p,
        'alpha': alpha,
        'lambda_max': lambda_max,
        'beta_c': beta_c,
        'eigenvalues': eigenvalues_real  # 保存所有特征值的实部
    })
    return lambda_max, beta_c

# 初始变量
N = 50
m = 5

mu = 0.1  # 恢复率
p_move_values = np.concatenate([
    np.arange(0.01, 0.1, 0.01),
    np.arange(0.1, 1.01, 0.01)
])

alpha_values = [0.4, 0.5, 0.6]  # 不同的alpha值

# 方法一：生成BA无标度网络
W = generate_ba_scale_free_network(N, m)

# 方法二：计算C矩阵
C = calculate_C_matrix(W)

# 方法三：生成k和n的值
n = np.full(N, 100)  # 每个子种群规模
k_values = np.full(N, 100)  # 平均度

# 记录所有中间变量的计算
all_results = []

# 计算在不同 alpha 和 p_move 下的beta_c和lambda_max
for alpha in alpha_values:
    for p_move in p_move_values:
        # 计算M^1和M^2
        M1 = calculate_M1_matrix(C, k_values, p_move)
        M2 = calculate_M2_matrix(C, n, p_move)
        
        # 计算总的M矩阵
        M = alpha * M1 + (1 - alpha) * M2
        
        # 保存中间矩阵M1和M2的最大值、最小值和均值
        record = {
            'alpha': alpha,
            'p_move': p_move,
            'M1_max': np.max(M1),
            'M1_min': np.min(M1),
            'M1_mean': np.mean(M1),
            'M2_max': np.max(M2),
            'M2_min': np.min(M2),
            'M2_mean': np.mean(M2)
        }
        
        # 计算特征值和beta_c
        lambda_max, beta_c = calculate_beta_c(M, mu, p_move, alpha, all_results)
        
        # 保存每步的中间值
        record['lambda_max'] = lambda_max
        record['beta_c'] = beta_c
        
        # 将每个 p_move 和 alpha 的计算结果记录
        all_results.append(record)

# 将所有中间变量和特征值结果保存为CSV文件
df = pd.DataFrame(all_results)
df.to_csv('/mnt/data/matrix_eigenvalues_and_intermediate_results.csv', index=False)

# 打印提示信息
print("中间变量、矩阵统计信息和特征值计算结果已保存到CSV文件中。")
